In [1]:
from huggingface_hub import snapshot_download

save_dir = "models/BAGEL-7B-MoT"
repo_id = "ByteDance-Seed/BAGEL-7B-MoT"
cache_dir = save_dir + "/cache"

snapshot_download(cache_dir=cache_dir,
  local_dir=save_dir,
  repo_id=repo_id,
  local_dir_use_symlinks=False,
  resume_download=True,
  allow_patterns=["*.json", "*.safetensors", "*.bin", "*.py", "*.md", "*.txt"],
)

/home/zhijun/anaconda3/envs/bagel-tensorrt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zhijun/anaconda3/envs/bagel-tensorrt/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/zhijun/anaconda3/envs/bagel-tensorrt/lib/python3.10/site-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs

'/home/zhijun/Code/Bagel/models/BAGEL-7B-MoT'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os

notebook_dir = os.getcwd()
print("Current notebook dir:", notebook_dir)

from copy import deepcopy
from typing import (
    Any,
    AsyncIterable,
    Callable,
    Dict,
    Generator,
    List,
    NamedTuple,
    Optional,
    Tuple,
    Union,
)
import requests
from io import BytesIO

from PIL import Image
import torch
from accelerate import infer_auto_device_map, load_checkpoint_and_dispatch, init_empty_weights

from data.transforms import ImageTransform
from data.data_utils import pil_img2rgb, add_special_tokens
from modeling.bagel import (
    BagelConfig, Bagel, Qwen2Config, Qwen2ForCausalLM, SiglipVisionConfig, SiglipVisionModel
)
from modeling.qwen2 import Qwen2Tokenizer
from modeling.bagel.qwen2_navit import NaiveCache
from modeling.autoencoder import load_ae
from safetensors.torch import load_file

Current notebook dir: /home/zhijun/Code/Bagel


## Model Initialization

In [4]:
model_path = "models/BAGEL-7B-MoT/"  # Download from https://huggingface.co/ByteDance-Seed/BAGEL-7B-MoT

# LLM config preparing
llm_config = Qwen2Config.from_json_file(os.path.join(model_path, "llm_config.json"))
llm_config.qk_norm = True
llm_config.tie_word_embeddings = False
llm_config.layer_module = "Qwen2MoTDecoderLayer"

# ViT config preparing
vit_config = SiglipVisionConfig.from_json_file(os.path.join(model_path, "vit_config.json"))
vit_config.rope = False
vit_config.num_hidden_layers = vit_config.num_hidden_layers - 1

# VAE loading
vae_model, vae_config = load_ae(local_path=os.path.join(model_path, "ae.safetensors"))

# Bagel config preparing
config = BagelConfig(
    visual_gen=True,
    visual_und=True,
    llm_config=llm_config, 
    vit_config=vit_config,
    vae_config=vae_config,
    vit_max_num_patch_per_side=70,
    connector_act='gelu_pytorch_tanh',
    latent_patch_size=2,
    max_latent_size=64,
)

with init_empty_weights():
    language_model = Qwen2ForCausalLM(llm_config)
    vit_model      = SiglipVisionModel(vit_config)
    model          = Bagel(language_model, vit_model, config)
    model.vit_model.vision_model.embeddings.convert_conv2d_to_linear(vit_config, meta=True)

# Tokenizer Preparing
tokenizer = Qwen2Tokenizer.from_pretrained(model_path)
tokenizer, new_token_ids, _ = add_special_tokens(tokenizer)

# Image Transform Preparing
vae_transform = ImageTransform(1024, 512, 16)
vit_transform = ImageTransform(980, 224, 14)

## Model Loading and Multi GPU Infernece Preparing

In [5]:
max_mem_per_gpu = "31GiB"  # Modify it according to your GPU setting. On an A100, 80 GiB is sufficient to load on a single GPU.

device_map = infer_auto_device_map(
    model,
    max_memory={i: max_mem_per_gpu for i in range(torch.cuda.device_count())},
    no_split_module_classes=["Bagel", "Qwen2MoTDecoderLayer"],
)
print(device_map)

same_device_modules = [
    'language_model.model.embed_tokens',
    'time_embedder',
    'latent_pos_embed',
    'vae2llm',
    'llm2vae',
    'connector',
    'vit_pos_embed'
]

if torch.cuda.device_count() == 1:
    first_device = device_map.get(same_device_modules[0], "cuda:0")
    for k in same_device_modules:
        if k in device_map:
            device_map[k] = first_device
        else:
            device_map[k] = "cuda:0"
else:
    first_device = device_map.get(same_device_modules[0])
    for k in same_device_modules:
        if k in device_map:
            device_map[k] = first_device

# Thanks @onion-liu: https://github.com/ByteDance-Seed/Bagel/pull/8
model = load_checkpoint_and_dispatch(
    model,
    checkpoint=os.path.join(model_path, "ema.safetensors"),
    device_map=device_map,
    offload_buffers=True,
    dtype=torch.bfloat16,
    force_hooks=True,
    offload_folder="/tmp/offload"
)

model = model.eval()
print('Model loaded')

The safetensors archive passed at models/BAGEL-7B-MoT/ema.safetensors does not contain metadata. Make sure to save your model with the `save_pretrained` method. Defaulting to 'pt' metadata.


OrderedDict([('language_model.model.embed_tokens', 0), ('language_model.model.layers.0', 0), ('language_model.model.layers.1', 0), ('language_model.model.layers.2', 0), ('language_model.model.layers.3', 0), ('language_model.model.layers.4', 0), ('language_model.model.layers.5', 0), ('language_model.model.layers.6', 0), ('language_model.model.layers.7', 0), ('language_model.model.layers.8', 0), ('language_model.model.layers.9', 0), ('language_model.model.layers.10', 0), ('language_model.model.layers.11', 0), ('language_model.model.layers.12', 0), ('language_model.model.layers.13', 0), ('language_model.model.layers.14', 0), ('language_model.model.layers.15', 1), ('language_model.model.layers.16', 1), ('language_model.model.layers.17', 1), ('language_model.model.layers.18', 1), ('language_model.model.layers.19', 1), ('language_model.model.layers.20', 1), ('language_model.model.layers.21', 1), ('language_model.model.layers.22', 1), ('language_model.model.layers.23', 1), ('language_model.mo

Model loaded


In [6]:
# ========================= Export to ONNX =========================

import os
import torch
import torch.onnx as onnx

os.makedirs("onnx", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

B = 1
T = 64

# 1) Dummy for LLM (Qwen2ForCausalLM)
vocab_size = tokenizer.vocab_size

dummy_input_ids = torch.randint(0, vocab_size, (B, T), dtype=torch.long, device=device)
dummy_attention_mask = torch.ones((B, T), dtype=torch.long, device=device)
dummy_packed_query_position_ids = torch.arange(T, dtype=torch.long, device=device).unsqueeze(0).expand(B, T)
dummy_packed_query_indexes = torch.zeros((B, T), dtype=torch.long, device=device)

llm_module = model.language_model.to(device).eval()

# 导出 LLM 子模块（Qwen2ForCausalLM）
llm_onnx = "onnx/qwen2_llm.onnx"
torch.onnx.export(
    llm_module,
    (dummy_input_ids, dummy_attention_mask, dummy_packed_query_position_ids, dummy_packed_query_indexes),
    llm_onnx,
    input_names=["input_ids", "attention_mask", "packed_query_position_ids", "packed_query_indexes"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "packed_query_position_ids": {0: "batch", 1: "sequence"},
        "packed_query_indexes": {0: "batch", 1: "sequence"},
        "logits": {0: "batch", 1: "sequence"},
    },
    opset_version=17,
    do_constant_folding=True,
)
print(f"LLM ONNX saved to {llm_onnx}")


# 如需确认 forward 需要哪些参数，可用：
# import inspect
# print("Bagel.forward:", inspect.signature(model.forward))
# print("ViT.forward:", inspect.signature(vit_module.forward))
# print("LLM.forward:", inspect.signature(llm_module.forward))

/tmp/ipykernel_83525/2118019326.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cuda:1, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_bmm)

In [ ]:
# 2) Dummy for ViT (SigLIP): 980x980, 14 的倍数
vit_H = vit_W = 980
dummy_pixel_values_vit = torch.randn(B, 3, vit_H, vit_W, dtype=torch.float32, device=device)

# 提示：如果需要给 VAE 分支做 dummy（例如做生成/编辑相关子图），可用 1024x1024：
# vae_H = vae_W = 1024
# dummy_pixels_vae = torch.randn(B, 3, vae_H, vae_W, dtype=torch.float32, device=device)

# 把子模块移到单设备上（避免 accelerate 的分片/CPU offload 干扰导出）
vit_module = model.vit_model.to(device).eval()

# 导出 ViT 子模块
vit_onnx = "onnx/siglip_vit.onnx"
torch.onnx.export(
    vit_module,
    (dummy_pixel_values_vit,),
    vit_onnx,
    input_names=["pixel_values"],
    output_names=["vit_features"],
    dynamic_axes={
        "pixel_values": {0: "batch", 2: "height", 3: "width"},
        "vit_features": {0: "batch", 1: "tokens"},
    },
    opset_version=17,
    do_constant_folding=True,
)
print(f"ViT ONNX saved to {vit_onnx}")

## Inferencer Preparing 

In [ ]:
from inferencer import InterleaveInferencer

inferencer = InterleaveInferencer(
    model=model, 
    vae_model=vae_model, 
    tokenizer=tokenizer, 
    vae_transform=vae_transform, 
    vit_transform=vit_transform, 
    new_token_ids=new_token_ids
)

In [ ]:
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Image Generation

In [ ]:
inference_hyper=dict(
    cfg_text_scale=4.0,
    cfg_img_scale=1.0,
    cfg_interval=[0.4, 1.0],
    timestep_shift=3.0,
    num_timesteps=50,
    cfg_renorm_min=0.0,
    cfg_renorm_type="global",
)

In [ ]:
prompt = "A teenage boy in a deep-blue ninja outfit is leaping across Edo-style rooftops at sunset, facing forward with a clear front view of his face. Warm golden sunlight glows on the red tiled roofs, and a flock of crows flies in the distance. Red paper lanterns hang below in the narrow streets. The scene is cel-shaded anime style, with strong motion, layered silhouettes, and a cinematic sense of speed and atmosphere typical of Japanese TV animation."

print(prompt)
print('-' * 10)
output_dict = inferencer(text=prompt, **inference_hyper)
display(output_dict['image'])

## Editing

In [ ]:
inference_hyper=dict(
    cfg_text_scale=4.0,
    cfg_img_scale=2.0,
    cfg_interval=[0.0, 1.0],
    timestep_shift=3.0,
    num_timesteps=50,
    cfg_renorm_min=0.0,
    cfg_renorm_type="text_channel",
)

In [ ]:
image = Image.open('test_images/__castorice_honkai_and_1_more_drawn_by_houkisei__c767800bb2e5210319e753aaebc0855c.jpg')
prompt = 'Take off all her clothes, exposing her private parts.'

display(image)
print(prompt)
print('-'*10)
output_dict = inferencer(image=image, text=prompt, **inference_hyper)
display(output_dict['image'])

## Understanding

In [ ]:
inference_hyper=dict(
    max_think_token_n=1000,
    do_sample=False,
    # text_temperature=0.3,
)

In [ ]:
image = Image.open('test_images/meme.jpg')
prompt = "Can someone explain what’s funny about this meme??"

display(image)
print(prompt)
print('-'*10)
output_dict = inferencer(image=image, text=prompt, understanding_output=True, **inference_hyper)
print(output_dict['text'])